# **Alerta temprana** de loops: ¿el modelo "avisa" antes de empezar a repetirse? — Qwen3.5-0.8B (Colab L4)

Hoy todas las señales de loop llegan tarde: el detector de texto se dispara cuando la mitad de los últimos 64 tokens ya se
repiten, y la dirección del parche / el probe llegan ~500–1,000 tokens después. Aquí buscamos una dirección en el
residual de los **512 tokens ANTES** de que se dispare el detector de texto (`wsi/precursor.py`).

Se compara, en respuestas apartadas (mitad B; la mitad A ajusta las direcciones y los umbrales):
- **precursor**: dirección "lo que viene antes del loop" vs. tokens normales, por capa
- **loop probe**: el probe de siempre (tokens ya en loop), con la misma prueba
- **controles**: dirección al azar, la posición del token sola (los loops pasan tarde) y un detector de texto más
  sensible (repetición en los últimos 64 / 256 tokens). La dirección tiene que ganarle a estos tres.

Umbral: que solo ~10 % de las respuestas normales de la mitad A den falsa alarma. Métrica clave: **% de loops avisados
antes del detector de texto** (hasta 2,048 tokens antes) y con cuántos tokens de adelanto.

Reusa las respuestas del base ya guardadas en Drive (GSM8K `discover_blind` y MATH-500 `discover_math500_qwen35-08b`).
No genera nada. ~15–30 min en L4. Resultado: `MyDrive/wsi_runs/FINALES/onset_precursor_qwen35-08b.json`.
Al final el runtime se apaga solo.

In [ ]:
MODEL = "Qwen/Qwen3.5-0.8B"
RUN_NAME = "onset_precursor_qwen35-08b"
RUNS = {"gsm8k": "discover_blind", "math": "discover_math500_qwen35-08b"}   # carpetas en MyDrive/wsi_runs
REPO = "/content/wsi"
BASE = "/content/base"
S = f"{REPO}/scripts/experiment_onset_precursor.py"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q hf_transfer
!pip install -q flash-linear-attention || echo "fla opcional"   # causal-conv1d compiles from source for ages: skip it
import torch, transformers; print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, glob, shutil, json, subprocess, datetime, time
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/wsi_runs"
FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
os.makedirs(FINALES, exist_ok=True); os.makedirs(OUTD, exist_ok=True)
WORK = {}
for ds, run in RUNS.items():
    src = f"{ROOT}/{run}"
    need = ("config.json", "requests_test.jsonl", "outputs_test_base.jsonl")
    if not all(os.path.exists(f"{src}/{f}") for f in need):
        print(f"{ds}: faltan archivos en {src} -> se salta")
        continue
    w = f"/content/work_{ds}"
    os.makedirs(w, exist_ok=True)
    for f in need:
        shutil.copy2(f"{src}/{f}", w)
    WORK[ds] = w
    print(ds, "<-", src)
assert WORK, "no hay ninguna corrida que reusar en Drive"

In [ ]:
if not os.path.exists(REPO):
    !git clone -q --depth 1 https://github.com/krexito/weight-space-index.git {REPO}
else:
    !cd {REPO} && git fetch -q --depth 1 origin main && git reset -q --hard origin/main
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
assert os.path.exists(S), "el repo no tiene experiment_onset_precursor.py: falta push"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=BASE)
print("commit", COMMIT)

In [ ]:
RES = {}
for ds, w in WORK.items():
    out = f"{OUTD}/{ds}.json"
    if not os.path.exists(out):                       # retoma: lo ya hecho no se repite
        print(f"\n===== {ds} =====", flush=True)
        !cd {REPO}/scripts && python -u {S} --work {w} --model {BASE} --out /content/pre_{ds}.json
        shutil.copy2(f"/content/pre_{ds}.json", out)
    RES[ds] = json.load(open(out))

In [ ]:
for ds, r in RES.items():
    print(f"\n{ds}: {r['n_half_b']} respuestas apartadas, {r['loop_rate_b']:.0%} con loop")
    rows = [(f"precursor L{r['best']['precursor']['layer']}", r["best"]["precursor"]),
            (f"loop probe L{r['best']['loopprobe']['layer']}", r["best"]["loopprobe"])] + list(r["baselines"].items())
    for name, v in rows:
        print(f"  {name:22s} AUC {v['auc']:.3f} | avisa antes del texto {v['warned_useful']} (adelanto mediano {v['median_lead_useful']} tok)"
              f" | demasiado pronto {v['warned_too_early']} | normales con alarma {v['normal_answers_fired']}")
out = {"runs": RES}
out["_final"] = {"complete": True, "model": MODEL, "commit": COMMIT, "notebook": "onset_precursor_colab.ipynb",
                 "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
                 "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w") as fh:
    json.dump(out, fh, indent=1)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en Drive:", dst)

In [ ]:
# Apagar el runtime: solo si el resultado final ya está en Drive
AUTO_APAGAR = True
dst = f"{FINALES}/{RUN_NAME}.json"
ok = os.path.exists(dst) and json.load(open(dst)).get("_final", {}).get("complete")
if AUTO_APAGAR and ok:
    print("Resultado en FINALES. Apagando el runtime en 60 s...", flush=True)
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en FINALES todavía.")